In [2]:
import pandas as pd
import numpy as np
import random
import pyomo.environ as pyo
from pyomo.environ import *
from pyomo.environ import SolverFactory
import yfinance as yf
from pathlib import Path
import os
import matplotlib.pyplot as plt


In [1]:
anos = ['2025-12-31',
 '2024-12-31',
 '2023-12-31',
 '2022-12-31',
 '2021-12-31',
 '2020-12-31',
 '2019-12-31',
 '2018-12-31',
 '2017-12-31',
 '2016-12-31',
 '2015-12-31',
 ]

In [12]:
# Retornos

dfs = {}
for ano in anos:
    ano_usado = ano.split('-')[0]
    nome = f'df_ativos_{ano_usado}.csv'
    # caminho = Path('retornos_ativos_otimizacao') / nome
    dfs[ano_usado] = pd.read_csv('../dados_retornos/otimizacao/ativos/'+nome)
    dfs[ano_usado]['date'] = pd.to_datetime(dfs[ano_usado]['date'])
    dfs[ano_usado] = dfs[ano_usado].set_index('date')
    dfs[ano_usado] = dfs[ano_usado].drop(columns=dfs[ano_usado].columns[(dfs[ano_usado] == 0).all()])
    print(ano,'-',len(dfs[ano_usado].columns))
    

2025-12-31 - 78
2024-12-31 - 78
2023-12-31 - 78
2022-12-31 - 77
2021-12-31 - 77
2020-12-31 - 73
2019-12-31 - 73
2018-12-31 - 70
2017-12-31 - 70
2016-12-31 - 67
2015-12-31 - 62


In [19]:
df = dfs['2025']
df_cov = df.cov()
df.head(2)

,ABEV3,ALOS3,ANIM3,AXIA3,AZZA3,B3SA3,BBAS3,BBDC3,BBDC4,BBSE3,...,TAEE11,TEND3,TOTS3,UGPA3,USIM5,VALE3,VBBR3,VIVT3,WEGE3,YDUQ3
date,,,,,,,,,,,,,,,,,,,,,
2025-10-01,-0.008268,-0.008518,-0.023326,0.005713,-0.019901,-0.023131,-0.007242,-0.019696,-0.016960,0.004511,...,0.007363,-0.001534,-0.024168,-0.034590,0.037825,0.012677,-0.017487,-0.005575,-0.021865,-0.013928
2025-10-02,-0.005011,-0.015227,-0.038790,-0.007953,-0.051440,-0.019096,-0.007752,-0.013595,-0.016177,-0.009879,...,-0.011643,-0.019191,-0.021195,-0.000474,-0.004556,0.006688,-0.013656,-0.018006,0.009500,-0.030610


In [22]:
score_todos_anos = pd.read_excel(f'../raspagem/piotroski/resultado/{2025}.xlsx')
score_usado_p = score_todos_anos[['Unnamed: 0', 'p_norm']].set_index('Unnamed: 0').T
print("TAMANHO DE ATIVOS NO SCORE: ",len(score_usado_p.columns))
print("TAMANHO DE ATIVOS NO df: ",len(df.columns))

ativos = df.columns.intersection(score_usado_p.columns)
df = df[ativos]
score = score_usado_p[ativos]
print("TAMANHO DE ATIVOS NO SCORE (2): ",len(score.columns))
print("TAMANHO DE ATIVOS NO df (2): ",len(df.columns))


TAMANHO DE ATIVOS NO SCORE:  89
TAMANHO DE ATIVOS NO df:  78
TAMANHO DE ATIVOS NO SCORE (2):  65
TAMANHO DE ATIVOS NO df (2):  65


In [30]:
df = (df - df.min())/(df.max()-df.min())

In [33]:
vb_cardinalidade_max = 10
vb_cardinalidade_min = 10
vb_peso_maximo = 0.20
vb_peso_minimo = 0.02
vb_theta = 0.5

In [41]:


#  --------- MODELO
model = pyo.ConcreteModel()
#---------VARIÁVEIS-----------
model.nome_ativos = pyo.Set(initialize = df.columns)
model.ativos = pyo.RangeSet(0, len(df.columns)-1)
model.dias = pyo.RangeSet(0, len(df)-1)
model.retornos_ativos = pyo.Param(model.dias, model.ativos, initialize=lambda model,dia, ativo: df.iloc[dia, ativo])    
model.sigma = pyo.Param(model.ativos, model.ativos, initialize=lambda model,a,b: df_cov.iloc[a,b])
# model.sigma_min = pyo.Param(initialize=0, mutable=True)
# model.sigma_max = pyo.Param(initialize=1, mutable=True)
model.score = pyo.Param(model.ativos, initialize=lambda model,a: score.iloc[0,a])
model.theta = pyo.Param(initialize=0.5)

# ------------------------------

#linearizar Sharpe
# z:= kx
model.z = pyo.Var(model.ativos, domain=pyo.NonNegativeReals)
model.k = pyo.Var(domain=pyo.NonNegativeReals)
model.y = pyo.Var(model.ativos, domain=pyo.Binary)
model.w = pyo.Var(model.ativos, domain=pyo.NonNegativeReals)
model.KMAX = pyo.Param(initialize=100)

# Com a problemática de cardinalidade y e bilinearidade, resolve-se
# PESO
# antes era model.x[a] <= model.pesomax * model.y[a]
#  z=kx  -> model.z[a] <= model.pesomax * modely[a] * model.k[a]
# CARDINALIDADE
# era sum(model.y[a] ...) >= e <=
# é feita a tranformação de w <= k * y

# Com y=0 
def res_y_zero(model,a):
        return model.w[a] <= model.KMAX * model.y[a]
model.y_zero = pyo.Constraint(model.ativos, rule=res_y_zero)

#--------------------------------------------------------------------------
# com y=1
#  segundo teto, quando y=1 ; w = k , pelas restriçoes >= e <=
def res_y_1_teto2(model,a):
        return model.w[a] <= model.k 
model.res_teto2 = pyo.Constraint(model.ativos, rule=res_y_1_teto2)

# Limite inf 1
def limite_1(model,a):
        return model.w[a] >= model.k - model.KMAX*(1-model.y[a])
model.r_limite1 = pyo.Constraint(model.ativos, rule=limite_1)
#--------------------------------------------------------------------------
#  PESOS MIN MAX
def pesomin(model,a):
        return model.z[a] >= 0.02 * model.w[a]
model.r_pesomin = pyo.Constraint(model.ativos, rule=pesomin)

def pesomax(model,a):
        return model.z[a] <= 0.2 * model.w[a]
model.r_pesomax = pyo.Constraint(model.ativos, rule=pesomax)

#  cardinalidade
def card_min(model):
        return sum(model.y[a] for a in model.ativos) >= vb_cardinalidade_min
model.r_cardmin = pyo.Constraint(rule=card_min)
def card_max(model):
        return sum(model.y[a] for a in model.ativos) <= vb_cardinalidade_max
model.r_card_max = pyo.Constraint(rule=card_max)

# Restrições Sharpe
def restricao_z(model):
        return sum(model.z[a] for a in model.ativos) == model.k
model.r_restricao_z = pyo.Constraint(rule=restricao_z)

def restricao_z_retorno(model):
        return sum(model.sigma[a,b]*model.z[a]*model.z[b] for a in model.ativos for b in model.ativos)  <= 1
model.r_res_z_retorno = pyo.Constraint(rule=restricao_z_retorno)

## OBJETIVOS:  1 para pegar z min, 2 para z max, 3 para normalização
def obj_1(model):
        var = sum(model.z[a] * model.retornos_ativos[d,a] for a in model.ativos for d in model.dias)
        score = sum(model.z[a] * model.score[a] for a in model.ativos)
        return var + score
model.obj1 = pyo.Objective(rule=obj_1, sense=pyo.maximize)

# ------------------- solver 1
opt = SolverFactory('cplex', executable='C:\\Program Files\\IBM\\ILOG\\CPLEX_Studio_Community222\\cplex\\bin\\x64_win64\\cplex.exe')
res = opt.solve(model,tee=True)
model.write('modelo_debug.lp', io_options={'symbolic_solver_labels': True})
# if res.opt.termination_condition  == pyo.TerminationCondition.optimal:
#     print(f"-----MODELO FOI RESOLVIDO {an}------------")
# else:
#     print(f"MODEO NAO FOI RESOLVIDO {an}----------")
melhor_pesos = {list(model.nome_ativos.data())[a]:(model.z[a].value/model.k.value) for a in model.ativos}
carteiras_anuais = {
'pesos':  melhor_pesos,
'z[a]': {a:model.z[a].value for a in model.ativos},
'score': {a:model.score[a] for a in model.ativos}
# 'sharpe_anual': s_lo*np.sqrt(252),
}




Welcome to IBM(R) ILOG(R) CPLEX(R) Interactive Optimizer Community Edition 22.2.0.0
  with Simplex, Mixed Integer & Barrier Optimizers
5725-A06 5725-A29 5724-Y48 5724-Y49 5724-Y54 5724-Y55 5655-Y21
Copyright IBM Corp. 1988, 2026.  All Rights Reserved.

Type 'help' for a list of available commands.
Type 'help' followed by a command name for more
information on commands.

CPLEX> Logfile 'cplex.log' closed.
Logfile 'C:\Users\Pichau\AppData\Local\Temp\tmp6g9i020i.cplex.log' open.
CPLEX> Problem 'C:\Users\Pichau\AppData\Local\Temp\tmpjkppzwr5.pyomo.lp' read.
Read time = 0.00 sec. (0.17 ticks)
CPLEX> Problem name         : C:\Users\Pichau\AppData\Local\Temp\tmpjkppzwr5.pyomo.lp
Objective sense      : Maximize
Variables            :     196  [Nneg: 131,  Binary: 65]
Objective nonzeros   :      65
Linear constraints   :     328  [Less: 326,  Greater: 1,  Equal: 1]
  Nonzeros           :     911
  RHS nonzeros       :      67
Quadratic constraints:       1  [Less: 1]
  Linear terms       :    

In [49]:
[{y:carteiras_anuais['pesos'][y]} for y in carteiras_anuais['pesos'] if carteiras_anuais['pesos'][y]>= 0.02]


[{'ALOS3': 0.02000005088649685},
 {'AXIA3': 0.09750174727421941},
 {'EQTL3': 0.14192535904453712},
 {'HYPE3': 0.06407318882182035},
 {'LREN3': 0.02000000034089044},
 {'MRVE3': 0.04160383948617135},
 {'PRIO3': 0.19999999854846595},
 {'SMFT3': 0.17215866623644044},
 {'TOTS3': 0.042737149803808515},
 {'YDUQ3': 0.1999999997251086}]

## -----------------------

In [ ]:
y = []
carteiras_anuais = {}
melhor_pesos = None
historico = []
carteiras_criadas = []
for a in anos:
    y.append(a.split('-')[0])
print("=-"*48)
print("Anos totais: ",y)
print("=-"*48)
lista_ativos_finais = {}
for an in anos:
    ano = an.split("-")[0]
    ano_int = int(ano)
    print("COMEÇANDO ANO NOVO: ",ano)
    #deletar modelo
    if 'model' in locals():
        del model
        print('Modelo Antigo deletado \n Iniciando Novo')
    else:
        print("Nao consta model")
        pass
    try:
        # retorno_usado = pd.DataFrame(eval(df_usado))
        df_usado = dfs[str(ano_int)]
        retorno_usado = df_usado.copy()

        # COV
        df_cov = retorno_usado.cov()

        # Lista ativos
        lista_ativos_finais[ano] = retorno_usado.columns.tolist()
    except Exception as e:
            print("=========ERRRRRRRRRRRROR")
            print(e)    

    print("# ------ CRIAÇÃO DO MODELO")
    print("## Carteira criada para o ano: ",ano_int+1)
    print("## UTILIZANDO DADOS DE RETORNO DE: ",ano)

    #  --------- MODELO
    model = pyo.ConcreteModel()
    #---------VARIÁVEIS-----------
    model.nome_ativos = pyo.Set(initialize = retorno_usado.columns)
    model.ativos = pyo.RangeSet(0, len(retorno_usado.columns)-1)
    model.dias = pyo.RangeSet(0, len(retorno_usado)-1)
    model.retornos_ativos = pyo.Param(model.dias, model.ativos, initialize=lambda model,dia, ativo: retorno_usado.iloc[dia, ativo])    
    model.sigma = pyo.Param(model.ativos, model.ativos, initialize=lambda model,a,b: df_cov.iloc[a,b])

    #linearizar Sharpe
    # z:= kx
    model.z = pyo.Var(model.ativos, domain=pyo.NonNegativeReals)
    model.k = pyo.Var(domain=pyo.NonNegativeReals)
    model.y = pyo.Var(model.ativos, domain=pyo.Binary)
    model.ky = pyo.Var(model.ativos, domain=pyo.NonNegativeReals)
    model.BIGM = pyo.Param(initialize=500)

    # Com a problemática de cardinalidade y e bilinearidade, resolve-se
    # PESO
    # antes era model.x[a] <= model.pesomax * model.y[a]
    #  z=kx  -> model.z[a] <= model.pesomax * model.y[a] * model.k
    # Criando uma auxiliar model.ky que representará model.y[a]*model.k

    # # Com y=0 eu eu preciso que model.ky seja 0
    def ky_zero(model,a):
        return model.ky[a] <= model.BIGM * model.y[a]
    model.y_zero = pyo.Constraint(model.ativos, rule=ky_zero)

    # com y=1 ou seja model.ky[a] <= model.y * model.k  (para y=1)
        #  segundo teto
    def res_y_1_teto2(model,a):
        return model.ky[a] <= model.k 
    model.res_teto2 = pyo.Constraint(model.ativos, rule=res_y_1_teto2)

        # Limite inf 1, se y é igual a 1, entao model.ky >= model.k   , e dessa forma faz o <= k e >= k , ou seja, ky=k para y=1
    def limite_1(model,a):
        return model.ky[a] >= model.k - model.BIGM*(1-model.y[a])
    model.r_limite1 = pyo.Constraint(model.ativos, rule=limite_1)

    #  PESOS MIN MAX
    def pesomin(model,a):
        return model.z[a] >= vb_peso_minimo * model.ky[a]
    model.r_pesomin = pyo.Constraint(model.ativos, rule=pesomin)

    def pesomax(model,a):
        return model.z[a] <= vb_peso_maximo * model.ky[a]
    model.r_pesomax = pyo.Constraint(model.ativos, rule=pesomax)

    #  cardinalidade
    def card_min(model):
        return sum(model.y[a] for a in model.ativos) >= vb_cardinalidade_min
    model.r_cardmin = pyo.Constraint(rule=card_min)
    def card_max(model):
        return sum(model.y[a] for a in model.ativos) <= vb_cardinalidade_max
    model.r_card_max = pyo.Constraint(rule=card_max)

    # Restrições Sharpe
    def restricao_z(model):
        return sum(model.z[a] for a in model.ativos) == model.k
    model.r_restricao_z = pyo.Constraint(rule=restricao_z)

    def restricao_z_retorno(model):
        return sum(model.z[a]*model.sigma[a,b]*model.z[b] for a in model.ativos for b in model.ativos) <= 1
    model.r_res_z_retorno = pyo.Constraint(rule=restricao_z_retorno)

    def obj_r(model):
        var = sum(sum(model.z[a]*model.retornos_ativos[t,a] for a in model.ativos)for t in model.dias)
        return var
    model.obj = pyo.Objective(rule=obj_r, sense=pyo.maximize)

    # ------------------- solver
    opt = SolverFactory('cplex', executable='C:\\Program Files\\IBM\\ILOG\\CPLEX_Studio_Community222\\cplex\\bin\\x64_win64\\cplex.exe')
    res = opt.solve(model,tee=False)

    melhor_pesos = {list(model.nome_ativos.data())[a]:(model.z[a].value/model.k.value) for a in model.ativos}
    carteiras_anuais[ano_int+1] = {
        'pesos':  melhor_pesos,
        # 'sharpe_anual': s_lo*np.sqrt(252),
    }
    print(f"{ano} -> {melhor_pesos}")
    print("+-="*30)
    if 'model' in locals():
        del model
        print('DELETADO DPS DO WHILE')
    else:
        print("Nao consta model")
        pass